# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

## Trade Level Analytics

In the previous notebook, we learned how to backtest the 'Head and Shoulders Pattern'. All the trades generated are saved in a CSV file `trades_hs_spy_1993_2018.csv`. You can download this file from the last unit of this course '**Python Codes and Data**'

In this notebook, we will learn how to evaluate the performance of a trading strategy. Simply looking at the overall profit or loss is not the most effective way to analyse a trading strategy. To assess if a strategy is viable, we must also analyse factors such as the time taken, number of trades, and average profit or loss on each trade. Therefore, in this notebook, we will learn how to calculate these metrics and evaluate the performance of trading the head and shoulders pattern. 

The key steps are:
1. [Read the Data](#read)
2. [Compute Trade Level Analytics](#trade)<br>
   2.1. [Profit and Loss](#pnl)<br>
   2.2. [Win Percentage](#win)<br>
   2.3. [Average PnL Per Trade](#avg)<br>
   2.4. [Average Trade Duration](#time)<br>
   2.5. [Profit Factor](#profit)<br>
3. [Function to Analyse the Trade Level Details](#function)

In [1]:
# For data manipulation
import pandas as pd
import numpy as np

# Ignore warnings
import warnings 
warnings.filterwarnings('ignore')

# For plotting
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

<a id='read'></a>
## Read the Data 

We have saved the trades data in a CSV file named `trades_hs_spy_1993_2018.csv`. You can read the file using the `pandas.read_csv()` method. 

In [2]:
# Import the trade sheet
trades = pd.read_csv('../data_modules/trades_hs_spy_1993_2018.csv', index_col=0)

# Print the top five rows
trades.head()

,entry_date,entry_price,position,exit_date,exit_type,exit_price,PnL
0,1993-11-04,26.91,-1.0,1993-12-22,SL,27.65,-0.74
0,2001-06-13,83.25,-1.0,2002-07-18,TP,59.53,23.72
0,2003-08-05,66.50,-1.0,2003-08-29,SL,69.96,-3.46
0,2006-02-07,90.53,-1.0,2006-02-16,SL,93.18,-2.65
0,2011-09-22,91.54,-1.0,2011-10-14,SL,99.42,-7.88


<a id='trade'></a>
## Compute Trade Level Analytics

In this section, we will calculate a few trade level analytics in order to analyse the trade level performance of the strategy.

First, we will create a dataframe named `analytics` to store all the metrics we calculate in this notebook. 

In [3]:
# Create dataframe to store trade analytics
analytics = pd.DataFrame(index=['Strategy'])

<a id='pnl'></a>
### Profit and Loss 
The profit and loss metric is nothing but the sum of all the gains and losses that were incurred on all the trades. 

We will calculate the PnL using the `sum()` function on the column named `PnL` in our dataset. This will tell us if our strategy incurred an overall profit or loss. 

In [4]:
# Calculate total PnL
analytics['Total PnL'] = trades.PnL.sum()

# Print the value
print("Total PnL: ", analytics['Total PnL'][0])

Total PnL:  14.05


The value returned is positive, which indicates that the strategy earned a total profit of around $14.05. However, we cannot measure a strategy's success just on the basis of `PnL`, as it provides no information regarding the number of trades. So, we will now calculate the win percentage to better evaluate the strategy.

<a id='win'></a>
### Win Percentage
The win percentage or win rate is an essential metric. It represents the percentage of trades which were profitable out of the total trades, to determine a strategy's success. A win rate above 50% is usually favourable.

Calculating the win percentage takes the number of profitable trades divided by the total number of trades as shown below. 

$$ Win~Rate = \frac{No.~of~Winning~Trades}{Total~No.~of~Trades} *100$$

Similarly, we can also calculate the Loss Percentage by taking the number of losing trades divided by the total number of trades. First, let's find the total number of trades we have made from the strategy. 

In [5]:
# Number of total trades
analytics['total_trades'] = len(trades.loc[trades.position== -1])

Next, we will find the number of winners and losers. Trades that gave us a profit i.e. `Pnl`>0 are winning trades. Trades that gave us a loss or no profit i.e. `Pnl`<=0 are losing trades. 

In [6]:
# Profitable trades
analytics['Number of Winners'] = len(trades.loc[trades.PnL>0])

# Loss-making trades
analytics['Number of Losers'] = len(trades.loc[trades.PnL<=0])

Now applying the formula discussed earlier, we can calculate the win and loss percentages, respectively. 

In [7]:
# Win percentage
analytics['Win (%)'] = 100*analytics['Number of Winners']/analytics.total_trades

# Loss percentage
analytics['Loss (%)'] = 100*analytics['Number of Losers']/analytics.total_trades

# Print trade level analytics
analytics.T

,Strategy
Total PnL,14.05
total_trades,8.00
Number of Winners,2.00
Number of Losers,6.00
Win (%),25.00
Loss (%),75.00


The win percentage works out to 25%. This indicates that around 75% of total trades were unsuccessful.

<b>Does this mean our strategy did not perform well?</b><br>

Not necessarily. A win rate around 50% or even below 50% does not always imply that our strategy failed to make money. 
It may sound logical that you can only make money if you have more winners than losers, but this is not always true. If your winners are giving much higher returns than your losers then you can still make good profits with a relatively low win rate. 

<a id='avg'></a>
### Average PnL Per Trade
The average PnL per trade is used to find the average amount that you can expect to gain or lose on each trade. This metric tells us how much impact a winning or losing trade might have. In general, we want the average loss per losing trade to be as low as possible and the average profit per winning trade as high as possible. For example, if your average loss per losing trade is 3x your average profit per winning trade then that means that a single loser will wipe out the profits of 3 winners.

You can determine the average profit per winning trade by dividing the sum amount of all the profits by the number of winning trades.

$$ Average~Profit~Per~Winning~Trade = \frac{Total~Profit~made~by~all~Winners}{No.~of~Winning~Trades} $$

Similarly, you can find the average loss per losing trade by dividing the sum of all the losses by the number of losing trades.


In [8]:
# Per trade profit/loss of winning trades
analytics['per_trade_PnL_winners'] = trades.loc[trades.PnL>0].PnL.mean()

# Per trade profit/loss of losing trades
analytics['per_trade_PnL_losers'] = np.abs(trades.loc[trades.PnL<=0].PnL.mean())

# Print trade level analytics
analytics.T

,Strategy
Total PnL,14.050
total_trades,8.000
Number of Winners,2.000
Number of Losers,6.000
Win (%),25.000
Loss (%),75.000
per_trade_PnL_winners,22.370
per_trade_PnL_losers,5.115


Here, you can see the average profit per trade comes out to be around 22.37 which is greater than the average loss per trade of around 5.115.

<a id='time'></a>
### Average Trade Duration
The average trade duration, also known as the average holding period, is the amount of time you remain in a trade on average. This is important because your capital is ‘locked’ during the time of the trade and cannot be used for other trades. That way it limits the number of trades you can take simultaneously. Hence, limiting your potential to increase profits. Another risk of a long holding period is the release of important news or earnings reports. One bad news can violently change the market and may result in a losing trade. 

Conversely, a short holding period may also not be favourable, as it results in higher transaction costs and may eat away your profits. 

To calculate the average trade duration we first calculate the holding period per trade i.e. `Exit Date` - `Entry Date`. Next, we calculate the mean of the holding time using the `mean()` method. 

In [9]:
# Convert entry time and exit time to datetime format
trades['entry_date'] = pd.to_datetime(trades['entry_date'])
trades['exit_date'] = pd.to_datetime(trades['exit_date'])

# Calculate holding period for each trade
holding_period = trades['exit_date'] - trades['entry_date']

# Calculate their mean
analytics['Average holding time'] = holding_period.mean()

# Print trade level analytics
analytics.T

,Strategy
Total PnL,14.05
total_trades,8
Number of Winners,2
Number of Losers,6
Win (%),25.0
Loss (%),75.0
per_trade_PnL_winners,22.37
per_trade_PnL_losers,5.115
Average holding time,75 days 00:00:00


We get the average holding period as 75 days.

<a id='profit'></a>
### Profit Factor
The profit factor measures the amount of money made against the money lost while trading. 
It is the ratio of the sum of profit to the sum of loss. It can also be calculated with the following formula: 

$$ Profit~Factor = \frac{~Win~Percentage~*~Average~Profit~Per~Winning~Trade}{~Loss~Percentage~*~Average~Loss~Per~Losing~Trade} $$

Ideally, a profit factor greater than 1 is desired. Anything below one is considered as unsatisfactory performance. There is a grading system for the profit factor to help you analyse the performance of your strategy. 

|S.No | Profit Factor | Interpretation    |
|---:|:-------------|:-----------|
| 1 | Below 1  | Strategy is unprofitable |
| 2 | Equal to 1  | Capital at the time of exit is same as capital at time of entry | 
| 3 | Between 1.10 and 1.40 | Strategy provides average returns, but may not withstand high volatility | 
| 4 | Between 1.40 and 2.0 | Strategy is decent | 
| 5 | Equal to or greater than 2  | Strategy is excellent | 

In [10]:
# Calculate profit factor
analytics['Profit Factor'] = (analytics['Win (%)']/100*analytics['per_trade_PnL_winners']) / \
    (analytics['Loss (%)']/100*analytics['per_trade_PnL_losers'])

# Print trade level analytics
analytics.T

,Strategy
Total PnL,14.05
total_trades,8
Number of Winners,2
Number of Losers,6
Win (%),25.0
Loss (%),75.0
per_trade_PnL_winners,22.37
per_trade_PnL_losers,5.115
Average holding time,75 days 00:00:00
Profit Factor,1.457804


Thus the profit factor comes out to be 1.46. This means our strategy has gained 1.46 dollars for every lost dollar.

<a id='function'></a>
## Function to Analyse the Trade Level Details

Let's create a function which will help us calculate the trade level analytics. This would enable us to call the function directly whenever we need to analyse the trade level details of the strategy. We will be using this function recurrently in the upcoming sections.

In [11]:
def trade_level_analytics(trades):
    # Create dataframe to store trade analytics
    analytics = pd.DataFrame(index=['Strategy'])

    # Calculate total PnL
    analytics['Total PnL'] = trades.PnL.sum()

    # Number of total trades
    analytics['total_trades'] = len(trades)

    # Profitable trades
    analytics['Number of Winners'] = len(trades.loc[trades.PnL > 0])

    # Loss-making trades
    analytics['Number of Losers'] = len(trades.loc[trades.PnL <= 0])

    # Win percentage
    analytics['Win (%)'] = 100 * analytics['Number of Winners'] / analytics.total_trades

    # Loss percentage
    analytics['Loss (%)'] = 100 * analytics['Number of Losers'] / analytics.total_trades

    # Per trade profit/loss of winning trades
    analytics['per_trade_PnL_winners'] = trades.loc[trades.PnL > 0].PnL.mean()

    # Per trade profit/loss of losing trades
    analytics['per_trade_PnL_losers'] = np.abs(trades.loc[trades.PnL <= 0].PnL.mean())

    # Convert entry time and exit time to datetime format
    trades['entry_date'] = pd.to_datetime(trades['entry_date'])
    trades['exit_date'] = pd.to_datetime(trades['exit_date'])

    # Calculate holding period for each trade
    holding_period = trades['exit_date'] - trades['entry_date']

    # Calculate their mean
    analytics['Average holding time'] = holding_period.mean()

    # Calculate profit factor
    analytics['Profit Factor'] = (analytics['Win (%)'] / 100 * analytics['per_trade_PnL_winners']) / (
            analytics['Loss (%)'] / 100 * analytics['per_trade_PnL_losers'])

    return analytics.T

In [12]:
# Call the function
trade_level_analytics(trades)

,Strategy
Total PnL,14.05
total_trades,8
Number of Winners,2
Number of Losers,6
Win (%),25.0
Loss (%),75.0
per_trade_PnL_winners,22.37
per_trade_PnL_losers,5.115
Average holding time,75 days 00:00:00
Profit Factor,1.457804


## Conclusion
In this notebook, we learned how to use a few trading metrics to analyse our trading strategy. These analytics help you find how the strategy has performed after the trade has been executed. However, it is also important to measure how the strategy has performed within the trade. We will evaluate this in the next notebook with the help of some performance metrics. <br><br>